In [ ]:
import pandas as pd
import numpy as np
import joblib
import sys
sys.path.append('../src')
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score
import matplotlib
matplotlib.use('Agg')
import shap


In [ ]:
# ── Load data ──────────────────────────────────────────
returns_with_events = pd.read_parquet('../data/returns_with_events.parquet')
log_returns         = pd.read_parquet('../data/log_returns.parquet')
price_levels        = pd.read_parquet('../data/price_levels.parquet')
try:
    extra_df = pd.read_parquet('../data/extra_features.parquet')
except:
    extra_df = None

symbols = returns_with_events['symbol'].unique().tolist()


In [ ]:
# ── Build features per stock ────────────────────────────
all_frames = []

for sym in symbols:
    df = returns_with_events[returns_with_events['symbol'] == sym].copy()
    df = df.sort_values('date').reset_index(drop=True)
    
    # Load NH-HMM state for this stock
    state_path = f'../data/per_stock_states/{sym}.parquet'
    try:
        states = pd.read_parquet(state_path)
        df = df.merge(states[['date','nh_state','nh_label']], on='date', how='left')
    except:
        df['nh_state'] = 1
        df['nh_label'] = 'Sideways/Neutral'
    
    if extra_df is not None and sym in extra_df.columns.levels[0]:
        sym_extra = extra_df[sym].copy()
        sym_extra.index.name = 'date'
        sym_extra.index = pd.to_datetime(sym_extra.index).tz_localize(None)
        sym_extra = sym_extra.reset_index()
        df['date'] = pd.to_datetime(df['date']).dt.tz_localize(None)
        df = df.merge(sym_extra, on='date', how='left')
    else:
        for col in ['volume_ratio', 'hl_range', 'rsi_14', 'dist_ma50']:
            df[col] = np.nan
    
    # Core features
    df['lag_return_1']  = df['log_return'].shift(1)
    df['lag_return_2']  = df['log_return'].shift(2)
    df['lag_return_5']  = df['log_return'].shift(5)
    df['momentum_5d']   = df['log_return'].rolling(5).sum().shift(1)
    df['momentum_20d']  = df['log_return'].rolling(20).sum().shift(1)
    df['vol_ratio']     = df['volatility'] / df['volatility'].rolling(20).mean()
    df['lag_vol_1']     = df['volatility'].shift(1)
    
    # Target: will tomorrow's return be positive?
    df['target'] = (df['log_return'].shift(-1) > 0).astype(int)
    
    all_frames.append(df)

full_df = pd.concat(all_frames, ignore_index=True)
full_df = full_df.dropna()

print(f"Total rows: {len(full_df)}")
print(f"Target distribution:\n{full_df['target'].value_counts(normalize=True)}")


In [ ]:
# ── Features list ───────────────────────────────────────
FEATURES = [
    'log_return', 'volatility', 'lag_return_1', 'lag_return_2',
    'lag_return_5', 'momentum_5d', 'momentum_20d', 'vol_ratio',
    'lag_vol_1', 'nh_state', 'E_t', 'days_to_event',
    'volume_ratio',
    'hl_range',
    'rsi_14',
    'dist_ma50'
]


In [ ]:
# ── Train/test split — time based ──────────────────────
split_date = pd.Timestamp('2022-06-01')
train = full_df[full_df['date'] < split_date]
test  = full_df[full_df['date'] >= split_date]

X_train = train[FEATURES]
y_train = train['target']
X_test  = test[FEATURES]
y_test  = test['target']

print(f"Train: {len(train)} rows | Test: {len(test)} rows")


In [ ]:
# ── Fit XGBoost ─────────────────────────────────────────
model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=10,   # prevents overfitting on small regimes
    random_state=42,
    eval_metric='logloss',
    early_stopping_rounds=20,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)


In [ ]:
# ── Evaluate ────────────────────────────────────────────
preds = model.predict(X_test)
probs = model.predict_proba(X_test)[:, 1]

print(f"\nTest Accuracy: {accuracy_score(y_test, preds):.4f}")
print(classification_report(y_test, preds, target_names=['Down', 'Up']))


In [ ]:
# ── SHAP feature importance ─────────────────────────────
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values, X_test, feature_names=FEATURES, plot_type='bar')


In [ ]:
# ── Save everything ─────────────────────────────────────
# Save model
joblib.dump(model, '../src/xgboost_classifier.pkl')

# Save features table (needed by app.py for live SHAP)
full_df['xgb_prob_up'] = model.predict_proba(full_df[FEATURES])[:, 1]
full_df[['date','symbol'] + FEATURES + ['target','xgb_prob_up']].to_parquet(
    '../data/xgb_features_full.parquet'
)

# Save SHAP values for test set (for static display in app)
shap_df = pd.DataFrame(shap_values, columns=FEATURES)
shap_df.to_parquet('../data/shap_values_test.parquet')

print("Saved xgboost_classifier.pkl, xgb_features_full.parquet, shap_values_test.parquet")
print(f"\nFeature importance:")
for f, imp in sorted(zip(FEATURES, model.feature_importances_), key=lambda x: -x[1]):
    print(f"  {f:25s}: {imp:.4f}")
